In [ ]:
from obspy.clients.fdsn import Client

from etna_config import (
    ETNA_WAVEFORM_CONFIG,
    ETNA_GAS_METEO_COLS,
    ETNA_EVENT_TIME,
)

from etna_waveform import build_station_waveform_dataset
from etna_dataset import (
    load_etnagas_csv,
    extract_plume_co2so2_xls,
    create_etna_final_dataset,
)
from etna_plotting_utils import (
    dataset_health_report,
    plot_scaled_dataset,
    compare_station_variable,
    plot_variable_pdfs,
    plot_raw_vs_scaled_pdfs,
    distribution_summary,
    run_teleseismic_checks,
    plot_final_dataset_with_event,
)

client = Client("INGV")

### Stations

In [ ]:
# identify the station metadata for ME01/ME02, get the exact vertical channel
inventory = client.get_stations(station="ME01" # or "ME02"
                                ,level="response",
                                starttime = "2008-04-12 00:00:00.000",endtime = "2008-05-13 00:00:00.000")
print(inventory)

In [ ]:
network = inventory[0]
station = network[0] 
num_channels = len(station)
print ('number of channels: ', num_channels)
print(station)

In [ ]:
channel = [c for c in station if c.code == "EHZ"][0]
print(channel)

### Fetch

In [ ]:
me01_wave, me01_failures = build_station_waveform_dataset("ME01")
me02_wave, me02_failures = build_station_waveform_dataset("ME02")

In [ ]:
# checkpoint
me01_wave.to_pickle("../etna_data/etna_me01_waveform.pkl")
me02_wave.to_pickle("../etna_data/etna_me02_waveform.pkl")

In [ ]:
# load from checkpoint
#me01_wave = pd.read_pickle("../etna_data/etna_me01_waveform.pkl")
#me02_wave = pd.read_pickle("../etna_data/etna_me02_waveform.pkl")

In [ ]:
# also save as CSV for easier inspection
#me01_wave.reset_index().to_csv("../etna_data/etna_me01_waveform.csv", index=False)
#me02_wave.reset_index().to_csv("../etna_data/etna_me02_waveform.csv", index=False)

## INGV & Final Dataset

### Other variables
adding
- WindSpeed 
- Patm_3 
- AirTemp_3 
- CO2_3 
- plume data (SO2/CO2 ratio)

In [ ]:
plume_df = extract_plume_co2so2_xls("../etna_data/1012-1_VolcanicGas_Etna.xls")

In [ ]:
ETNA_COLS = ["CO2_3", "AirTemp_3", "Patm_3", "WindSpeed"]

etnagas_df = pd.read_csv("../etna_data/3c.csv").replace("NULL", np.nan)
etnagas_df["timestamp"] = pd.to_datetime(etnagas_df["Time"], utc=True, errors="coerce")

etnagas_df = (
    etnagas_df[["timestamp"] + ETNA_COLS]
    .dropna(subset=["timestamp"])
    .sort_values("timestamp")
    .drop_duplicates(subset=["timestamp"], keep="first")
    .reset_index(drop=True)
)

### Merge and save final dataset

In [ ]:
ETNA_COLS = ["CO2_3", "AirTemp_3", "Patm_3", "WindSpeed"]

final_me01 = create_final_dataset(
    wave_df=me01_wave,
    station_name="ME01",
    out_csv="../etna_data/FINAL_ME01.csv",
    etnagas_df=etnagas_df,
    etnagas_cols=ETNA_COLS,
    plume_df=plume_df,
)

final_me02 = create_final_dataset(
    wave_df=me02_wave,
    station_name="ME02",
    out_csv="../etna_data/FINAL_ME02.csv",
    etnagas_df=etnagas_df,
    etnagas_cols=ETNA_COLS,
    plume_df=plume_df,
)

In [ ]:
# load from checkpoint
#final_me01 = pd.read_pickle("../etna_data/final_me01.pkl")
#final_me02 = pd.read_pickle("../etna_data/final_me02.pkl")

## Checks

In [ ]:
final_me01_raw, final_me01_scaled = final_me01
final_me02_raw, final_me02_scaled = final_me02

In [ ]:
station_data = {
    "ME01": {
        "wave": me01_wave,
        "final_raw": final_me01_raw,
        "final_scaled": final_me01_scaled,
    },
    "ME02": {
        "wave": me02_wave,
        "final_raw": final_me02_raw,
        "final_scaled": final_me02_scaled,
    },
}

#### Plot scaled variables for both stations

In [ ]:
for sta, d in station_data.items():
    plot_scaled_dataset(d["final_scaled"], sta)

#### Compare ME01 and ME02 directly in one figure

In [ ]:

for var in ["S_log_scaled", "T_log_scaled", "Y_log_scaled"]:
    if var in final_me01_scaled.columns and var in final_me02_scaled.columns:
        compare_station_variable(final_me01_scaled, final_me02_scaled, var)


#### PDF / distribution plots

In [ ]:
plot_variable_pdfs(final_me01_scaled, "ME01 hourly scaled")
plot_variable_pdfs(final_me02_scaled, "ME02 hourly scaled")

#### raw-vs-scaled PDFs

In [ ]:
variable_pairs = [
    ("S_log", "S_log_scaled"),
    ("T_log", "T_log_scaled"),
    ("Y_log", "Y_log_scaled"),
    ("CO2_3", "CO2_3_scaled"),
    ("AirTemp_3", "AirTemp_3_scaled"),
    ("Patm_3", "Patm_3_scaled"),
    ("WindSpeed", "WindSpeed_scaled"),
    ("CO2_SO2", "CO2_SO2_scaled"),
]

plot_raw_vs_scaled_pdfs(
    final_me01_raw,
    final_me01_scaled,
    "ME01 hourly",
    variable_pairs,
)

plot_raw_vs_scaled_pdfs(
    final_me02_raw,
    final_me02_scaled,
    "ME02 hourly",
    variable_pairs,
)

#### distribution summary table

In [ ]:

summary_me01 = distribution_summary(final_me01_scaled, "ME01 hourly scaled")
summary_me02 = distribution_summary(final_me02_scaled, "ME02 hourly scaled")

#### Inspecting teleseismic wave arrival considering Wenchuan earthquake event time and comparing 1-min vs 1h time grid

In [ ]:
results = {}
for station in ["ME01", "ME02"]:
    results[station] = run_teleseismic_checks(station)


In [ ]:
plot_final_dataset_with_event(
    "../etna_data/FINAL_ME01_scaled.csv",
    station="ME01",
    event_time=event_time,
)